# NB05 — Statistical Significance

**GraphSentry: A Unified GNN Framework for Blockchain Illicit Activity Detection**

Runs GraphSentry and all baselines across 5 random seeds to quantify variance
and test whether performance differences are statistically significant.

**Models:**
1. **GraphSentry** — GCN + CC-aware GraphSAINT sampling (NB02)
2. **GCN baseline** — same architecture, standard DataLoader (NB03)
3. **GAT baseline** — 4-head attention, standard DataLoader (NB03)
4. **GraphSAGE baseline** — mean aggregation, standard DataLoader (NB03)

**Seeds:** 42, 123, 456, 789, 1024

**Tests:** Paired t-test (AUROC) + Cohen's d effect size for each baseline vs GraphSentry.

**Requires:** Artefacts from `NB01_data_pipeline.ipynb`

**Produces:**
- `significance_results.json` — per-seed metrics + statistical tests

## 1. Configuration

In [47]:
MAX_EPOCHS = 60
PATIENCE = 10
BATCH_SIZE = 64
LR = 0.005
LR_FACTOR = 0.5
LR_PATIENCE = 5
HIDDEN_DIM = 128
DROPOUT = 0.5

SAINT_STRATEGY = 'node'
SAINT_BUDGET = 500
SAINT_SAMPLES_PER_EPOCH = 20

SEEDS = [42, 123, 456, 789, 1024]

BASE_PATH = '/content/drive/MyDrive/GraphSentry'
PROCESSED_PATH = f'{BASE_PATH}/data/processed'

## 2. Environment + load artefacts

In [48]:
!pip install -q uv
!uv pip install --system torch torch-geometric numpy pandas tqdm scikit-learn scipy

from google.colab import drive
import os, json, time, random, copy

import torch
import torch.nn.functional as F
import numpy as np
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, global_mean_pool
from torch_geometric.utils import degree
from torch_geometric.loader import DataLoader
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    confusion_matrix, average_precision_score
)
from scipy import stats

drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

bg = torch.load(os.path.join(PROCESSED_PATH, 'background_graph.pt'), weights_only=False)
meta = torch.load(os.path.join(PROCESSED_PATH, 'cc_metadata.pt'), weights_only=False)
feat_anon = torch.load(os.path.join(PROCESSED_PATH, 'features_anonymous.pt'), weights_only=False)

with open(os.path.join(PROCESSED_PATH, 'dataset_stats.json'), 'r') as f:
    dataset_stats = json.load(f)

INPUT_DIM = dataset_stats['feature_dims']['anonymous'] + 1
print(f"Dataset: {dataset_stats['total_ccs']} CCs, {dataset_stats['total_nodes']} nodes, input dim = {INPUT_DIM}")

Using Python 3.12.13 environment at: /usr
Checked 7 packages in 91ms
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Dataset: 5500 CCs, 39819 nodes, input dim = 44


## 3. Graph construction + data loaders

In [49]:
def build_pyg_graph(cc_id):
    cc_nodes = sorted(bg['cc_to_nodes'][cc_id])
    local_map = {g: l for l, g in enumerate(cc_nodes)}
    adj = bg['adj_list']

    x = feat_anon[cc_nodes].clone()

    node_set = set(cc_nodes)
    local_src, local_dst = [], []
    for g_src in cc_nodes:
        for g_dst in adj[g_src]:
            if g_dst in node_set and g_dst in local_map:
                local_src.append(local_map[g_src])
                local_dst.append(local_map[g_dst])

    if len(local_src) == 0:
        edge_index = torch.tensor([[0], [0]], dtype=torch.long)
    else:
        edge_index = torch.tensor([local_src, local_dst], dtype=torch.long)

    deg = degree(edge_index[1], x.size(0), dtype=torch.float)
    deg = torch.log(deg + 1).view(-1, 1)
    x = torch.cat([x, deg], dim=1)

    y = torch.tensor([meta['cc_label_map'][cc_id]], dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y)


val_data = [build_pyg_graph(cc_id) for cc_id in meta['val_ids']]
test_data = [build_pyg_graph(cc_id) for cc_id in meta['test_ids']]
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

# DataLoader training set (for baselines)
train_data = [build_pyg_graph(cc_id) for cc_id in meta['train_ids']]
train_pos = [d for d in train_data if d.y.item() == 1]
train_neg = [d for d in train_data if d.y.item() == 0]
oversample_factor = max(1, len(train_neg) // len(train_pos))
train_balanced = train_neg + train_pos * oversample_factor
train_loader = DataLoader(train_balanced, batch_size=BATCH_SIZE, shuffle=True)

print(f"Val: {len(val_data)}, Test: {len(test_data)}, Train: {len(train_balanced)} (oversampled)")

Val: 825, Test: 825, Train: 7000 (oversampled)


## 4. Model architectures

In [50]:
class SubgraphGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden=HIDDEN_DIM, dropout=DROPOUT):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.bn1 = torch.nn.BatchNorm1d(hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.bn2 = torch.nn.BatchNorm1d(hidden)
        self.conv3 = GCNConv(hidden, hidden)
        self.bn3 = torch.nn.BatchNorm1d(hidden)
        self.classifier = torch.nn.Linear(hidden, 2)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.bn3(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x)


class SubgraphGAT(torch.nn.Module):
    def __init__(self, in_channels, hidden=HIDDEN_DIM, heads=4, dropout=DROPOUT):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden // heads, heads=heads)
        self.bn1 = torch.nn.BatchNorm1d(hidden)
        self.conv2 = GATConv(hidden, hidden // heads, heads=heads)
        self.bn2 = torch.nn.BatchNorm1d(hidden)
        self.conv3 = GATConv(hidden, hidden, heads=1)
        self.bn3 = torch.nn.BatchNorm1d(hidden)
        self.classifier = torch.nn.Linear(hidden, 2)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.bn3(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x)


class SubgraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden=HIDDEN_DIM, dropout=DROPOUT):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden)
        self.bn1 = torch.nn.BatchNorm1d(hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.bn2 = torch.nn.BatchNorm1d(hidden)
        self.conv3 = SAGEConv(hidden, hidden)
        self.bn3 = torch.nn.BatchNorm1d(hidden)
        self.classifier = torch.nn.Linear(hidden, 2)
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = self.bn3(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.classifier(x)

## 5. GraphSAINT sampler

In [51]:
class SubgraphSAINTSampler:
    def __init__(self, eligible_cc_ids):
        self.eligible_ccs = set(eligible_cc_ids)
        self.N = bg['num_nodes']
        self.M = bg['num_edges']

        self.eligible_illicit = [c for c in self.eligible_ccs if meta['cc_label_map'][c] == 1]
        self.eligible_licit = [c for c in self.eligible_ccs if meta['cc_label_map'][c] == 0]
        self.cc_sizes = {cc: len(nodes) for cc, nodes in bg['cc_to_nodes'].items()}

    def sample(self, budget, strategy='node'):
        if strategy == 'node':
            sampled = set(random.sample(range(self.N), min(budget, self.N)))
        elif strategy == 'edge':
            ei = bg['edge_index']
            indices = random.sample(range(self.M), min(budget, self.M))
            sampled = set()
            for idx in indices:
                sampled.add(ei[0, idx].item())
                sampled.add(ei[1, idx].item())
        elif strategy == 'rw':
            adj = bg['adj_list']
            roots = random.sample(range(self.N), min(budget, self.N))
            sampled = set(roots)
            for root in roots:
                current = root
                for _ in range(5):
                    neighbors = adj[current]
                    if not neighbors:
                        break
                    current = random.choice(neighbors)
                    sampled.add(current)
        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        node_to_cc = bg['node_to_cc']
        touched = {node_to_cc[n] for n in sampled if n in node_to_cc and node_to_cc[n] in self.eligible_ccs}

        if not touched:
            touched = set(random.sample(list(self.eligible_ccs), min(32, len(self.eligible_ccs))))

        touched_illicit = [c for c in touched if meta['cc_label_map'][c] == 1]
        touched_licit = [c for c in touched if meta['cc_label_map'][c] == 0]

        if touched_illicit and touched_licit:
            factor = max(1, len(touched_licit) // len(touched_illicit))
            cc_ids = touched_licit + touched_illicit * factor
        elif not touched_illicit:
            n_inject = max(1, len(touched_licit) // 10)
            injected = random.choices(self.eligible_illicit, k=min(n_inject, len(self.eligible_illicit)))
            cc_ids = touched_licit + injected
        else:
            cc_ids = list(touched)

        data_list = [build_pyg_graph(cc_id) for cc_id in cc_ids]
        norm_weights = self._compute_norms(cc_ids, budget, strategy)
        batch = Batch.from_data_list(data_list)
        return batch, norm_weights

    def _compute_norms(self, cc_ids, budget, strategy):
        weights = []
        for cc_id in cc_ids:
            if strategy == 'node':
                p = 1 - (1 - self.cc_sizes[cc_id] / self.N) ** budget
            elif strategy == 'edge':
                d_c = sum(len(bg['adj_list'][n]) for n in bg['cc_to_nodes'][cc_id])
                p = 1 - (1 - d_c / (2 * self.M + 1)) ** budget
            elif strategy == 'rw':
                p = 1 - (1 - self.cc_sizes[cc_id] / self.N) ** (budget * 5)
            else:
                p = 1.0
            weights.append(1.0 / max(p, 1e-6))
        w = torch.tensor(weights, dtype=torch.float)
        return w * len(w) / w.sum()


def weighted_cross_entropy(logits, targets, norm_weights):
    per_sample = F.cross_entropy(logits, targets, reduction='none')
    return (per_sample * norm_weights.to(logits.device)).mean()


sampler = SubgraphSAINTSampler(meta['train_ids'])
print(f"Sampler ready: {len(sampler.eligible_ccs)} eligible CCs")

Sampler ready: 3850 eligible CCs


## 6. Evaluation + training functions

In [52]:
def evaluate(model, loader):
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            prob = F.softmax(out, dim=1)[:, 1]
            y_true.extend(batch.y.cpu().numpy())
            y_prob.extend(prob.cpu().numpy())

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    has_both = len(np.unique(y_true)) > 1
    return {
        'auroc': roc_auc_score(y_true, y_prob) if has_both else 0.0,
        'f1': f1_score(y_true, (y_prob >= 0.5).astype(int), zero_division=0),
        'y_true': y_true,
        'y_prob': y_prob,
    }


def tune_threshold(y_true, y_prob):
    best_f1, best_t = 0, 0.5
    for t in np.arange(0.1, 1.0, 0.05):
        f = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t, best_f1


def train_graphsentry(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = SubgraphGCN(INPUT_DIM).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE
    )

    best_val_auroc = 0
    best_state = None
    no_improve = 0

    t0 = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        epoch_loss = 0
        for _ in range(SAINT_SAMPLES_PER_EPOCH):
            batch, norm_weights = sampler.sample(SAINT_BUDGET, SAINT_STRATEGY)
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = weighted_cross_entropy(out, batch.y, norm_weights)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / SAINT_SAMPLES_PER_EPOCH
        val_m = evaluate(model, val_loader)
        scheduler.step(val_m['auroc'])

        if val_m['auroc'] > best_val_auroc:
            best_val_auroc = val_m['auroc']
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            break

    t_elapsed = time.time() - t0
    model.load_state_dict(best_state)

    val_r = evaluate(model, val_loader)
    best_thresh, _ = tune_threshold(val_r['y_true'], val_r['y_prob'])

    test_r = evaluate(model, test_loader)
    y_pred = (test_r['y_prob'] >= best_thresh).astype(int)

    return {
        'test_auroc': round(test_r['auroc'], 4),
        'test_auc_pr': round(average_precision_score(test_r['y_true'], test_r['y_prob']), 4),
        'test_f1': round(f1_score(test_r['y_true'], y_pred, zero_division=0), 4),
        'threshold': best_thresh,
        'training_time_s': round(t_elapsed, 1),
    }


def train_baseline(model_cls, name, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = model_cls(INPUT_DIM).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=LR_FACTOR, patience=LR_PATIENCE
    )

    best_val_auroc = 0
    best_state = None
    no_improve = 0

    t0 = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        epoch_loss = 0
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = F.cross_entropy(out, batch.y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        val_m = evaluate(model, val_loader)
        scheduler.step(val_m['auroc'])

        if val_m['auroc'] > best_val_auroc:
            best_val_auroc = val_m['auroc']
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            break

    t_elapsed = time.time() - t0
    model.load_state_dict(best_state)

    val_r = evaluate(model, val_loader)
    best_thresh, _ = tune_threshold(val_r['y_true'], val_r['y_prob'])

    test_r = evaluate(model, test_loader)
    y_pred = (test_r['y_prob'] >= best_thresh).astype(int)

    return {
        'test_auroc': round(test_r['auroc'], 4),
        'test_auc_pr': round(average_precision_score(test_r['y_true'], test_r['y_prob']), 4),
        'test_f1': round(f1_score(test_r['y_true'], y_pred, zero_division=0), 4),
        'threshold': best_thresh,
        'training_time_s': round(t_elapsed, 1),
    }

## 7. Run all models across 5 seeds

In [53]:
all_runs = {
    'GraphSentry': [],
    'GCN': [],
    'GAT': [],
    'SAGE': [],
}

for i, seed in enumerate(SEEDS):
    print(f"\n{'='*70}")
    print(f"Seed {seed} ({i+1}/{len(SEEDS)})")
    print(f"{'='*70}")

    print(f"  GraphSentry (GraphSAINT)...", end=' ')
    r = train_graphsentry(seed)
    all_runs['GraphSentry'].append(r)
    print(f"AUROC={r['test_auroc']:.4f}, F1={r['test_f1']:.4f}, {r['training_time_s']}s")

    for name, cls in [('GCN', SubgraphGCN), ('GAT', SubgraphGAT), ('SAGE', SubgraphSAGE)]:
        print(f"  {name} (DataLoader)...", end=' ')
        r = train_baseline(cls, name, seed)
        all_runs[name].append(r)
        print(f"AUROC={r['test_auroc']:.4f}, F1={r['test_f1']:.4f}, {r['training_time_s']}s")

print(f"\nAll runs complete.")


Seed 42 (1/5)
  GraphSentry (GraphSAINT)... AUROC=0.8881, F1=0.5079, 57.7s
  GCN (DataLoader)... AUROC=0.8525, F1=0.4105, 11.3s
  GAT (DataLoader)... AUROC=0.8821, F1=0.4796, 18.1s
  SAGE (DataLoader)... AUROC=0.8650, F1=0.4512, 10.9s

Seed 123 (2/5)
  GraphSentry (GraphSAINT)... AUROC=0.8686, F1=0.4560, 47.6s
  GCN (DataLoader)... AUROC=0.8608, F1=0.4386, 11.4s
  GAT (DataLoader)... AUROC=0.8803, F1=0.4481, 15.7s
  SAGE (DataLoader)... AUROC=0.8338, F1=0.4000, 10.1s

Seed 456 (3/5)
  GraphSentry (GraphSAINT)... AUROC=0.8679, F1=0.4198, 34.6s
  GCN (DataLoader)... AUROC=0.8885, F1=0.4382, 13.0s
  GAT (DataLoader)... AUROC=0.8910, F1=0.4609, 17.2s
  SAGE (DataLoader)... AUROC=0.8607, F1=0.4595, 9.2s

Seed 789 (4/5)
  GraphSentry (GraphSAINT)... AUROC=0.8734, F1=0.4381, 54.4s
  GCN (DataLoader)... AUROC=0.8759, F1=0.4706, 13.9s
  GAT (DataLoader)... AUROC=0.8760, F1=0.5065, 29.0s
  SAGE (DataLoader)... AUROC=0.8790, F1=0.4854, 10.9s

Seed 1024 (5/5)
  GraphSentry (GraphSAINT)... AUROC=0

## 8. Summary statistics

In [54]:
print("=" * 80)
print("SUMMARY: Mean ± Std across 5 seeds")
print("=" * 80)
print(f"{'Model':<20s} {'AUROC':>14s} {'AUC-PR':>14s} {'F1':>14s}")
print("-" * 65)

summary = {}
for model_name, runs in all_runs.items():
    aurocs = [r['test_auroc'] for r in runs]
    auc_prs = [r['test_auc_pr'] for r in runs]
    f1s = [r['test_f1'] for r in runs]

    summary[model_name] = {
        'auroc_mean': round(np.mean(aurocs), 4),
        'auroc_std': round(np.std(aurocs, ddof=1), 4),
        'auc_pr_mean': round(np.mean(auc_prs), 4),
        'auc_pr_std': round(np.std(auc_prs, ddof=1), 4),
        'f1_mean': round(np.mean(f1s), 4),
        'f1_std': round(np.std(f1s, ddof=1), 4),
        'aurocs': aurocs,
        'f1s': f1s,
    }

    s = summary[model_name]
    print(f"{model_name:<20s} "
          f"{s['auroc_mean']:.4f} ± {s['auroc_std']:.4f} "
          f"{s['auc_pr_mean']:.4f} ± {s['auc_pr_std']:.4f} "
          f"{s['f1_mean']:.4f} ± {s['f1_std']:.4f}")

print("-" * 65)

SUMMARY: Mean ± Std across 5 seeds
Model                         AUROC         AUC-PR             F1
-----------------------------------------------------------------
GraphSentry          0.8760 ± 0.0088 0.4925 ± 0.0115 0.4521 ± 0.0337
GCN                  0.8760 ± 0.0202 0.4933 ± 0.0486 0.4586 ± 0.0478
GAT                  0.8826 ± 0.0055 0.5323 ± 0.0324 0.4821 ± 0.0288
SAGE                 0.8655 ± 0.0210 0.4851 ± 0.0404 0.4503 ± 0.0311
-----------------------------------------------------------------


## 9. Statistical tests

In [55]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled_std = np.sqrt(((na - 1) * np.std(a, ddof=1)**2 + (nb - 1) * np.std(b, ddof=1)**2) / (na + nb - 2))
    if pooled_std == 0:
        return 0.0
    return (np.mean(a) - np.mean(b)) / pooled_std


gs_aurocs = summary['GraphSentry']['aurocs']

print("=" * 80)
print("STATISTICAL SIGNIFICANCE: GraphSentry vs each baseline")
print("=" * 80)
print(f"{'Comparison':<30s} {'Mean Δ':>8s} {'t-stat':>8s} {'p-value':>10s} {'Cohen d':>9s} {'Sig?':>6s}")
print("-" * 75)

sig_results = {}
for baseline in ['GCN', 'GAT', 'SAGE']:
    bl_aurocs = summary[baseline]['aurocs']

    t_stat, p_value = stats.ttest_rel(gs_aurocs, bl_aurocs)
    d = cohens_d(gs_aurocs, bl_aurocs)
    mean_delta = np.mean(gs_aurocs) - np.mean(bl_aurocs)
    sig = 'Yes' if p_value < 0.05 else 'No'

    sig_results[f'GraphSentry_vs_{baseline}'] = {
        'mean_delta': round(mean_delta, 4),
        't_statistic': round(t_stat, 4),
        'p_value': round(p_value, 6),
        'cohens_d': round(d, 4),
        'significant_at_005': p_value < 0.05,
    }

    print(f"  GS vs {baseline:<22s} {mean_delta:>+8.4f} {t_stat:>8.3f} {p_value:>10.6f} {d:>9.3f} {sig:>6s}")

print("-" * 75)
print()
print("Effect size interpretation: |d| < 0.2 negligible, 0.2-0.5 small,")
print("                           0.5-0.8 medium, > 0.8 large")

STATISTICAL SIGNIFICANCE: GraphSentry vs each baseline
Comparison                       Mean Δ   t-stat    p-value   Cohen d   Sig?
---------------------------------------------------------------------------
  GS vs GCN                     -0.0000   -0.004   0.997126    -0.003     No
  GS vs GAT                     -0.0066   -1.317   0.258134    -0.899     No
  GS vs SAGE                    +0.0105    1.281   0.269287     0.649     No
---------------------------------------------------------------------------

Effect size interpretation: |d| < 0.2 negligible, 0.2-0.5 small,
                           0.5-0.8 medium, > 0.8 large


## 10. Per-seed detail table

In [56]:
print("=" * 80)
print("PER-SEED AUROC")
print("=" * 80)
print(f"{'Seed':>6s}  {'GraphSentry':>12s} {'GCN':>8s} {'GAT':>8s} {'SAGE':>8s}")
print("-" * 50)

for i, seed in enumerate(SEEDS):
    print(f"{seed:>6d}  "
          f"{all_runs['GraphSentry'][i]['test_auroc']:>12.4f} "
          f"{all_runs['GCN'][i]['test_auroc']:>8.4f} "
          f"{all_runs['GAT'][i]['test_auroc']:>8.4f} "
          f"{all_runs['SAGE'][i]['test_auroc']:>8.4f}")

print("-" * 50)
print(f"{'Mean':>6s}  "
      f"{summary['GraphSentry']['auroc_mean']:>12.4f} "
      f"{summary['GCN']['auroc_mean']:>8.4f} "
      f"{summary['GAT']['auroc_mean']:>8.4f} "
      f"{summary['SAGE']['auroc_mean']:>8.4f}")
print(f"{'Std':>6s}  "
      f"{summary['GraphSentry']['auroc_std']:>12.4f} "
      f"{summary['GCN']['auroc_std']:>8.4f} "
      f"{summary['GAT']['auroc_std']:>8.4f} "
      f"{summary['SAGE']['auroc_std']:>8.4f}")

PER-SEED AUROC
  Seed   GraphSentry      GCN      GAT     SAGE
--------------------------------------------------
    42        0.8881   0.8525   0.8821   0.8650
   123        0.8686   0.8608   0.8803   0.8338
   456        0.8679   0.8885   0.8910   0.8607
   789        0.8734   0.8759   0.8760   0.8790
  1024        0.8819   0.9024   0.8834   0.8891
--------------------------------------------------
  Mean        0.8760   0.8760   0.8826   0.8655
   Std        0.0088   0.0202   0.0055   0.0210


## 11. Save results

In [57]:
output = {
    'seeds': SEEDS,
    'per_seed_runs': {k: v for k, v in all_runs.items()},
    'summary': {k: {sk: sv for sk, sv in v.items() if sk not in ('aurocs', 'f1s')}
                for k, v in summary.items()},
    'significance_tests': sig_results,
    'config': {
        'max_epochs': MAX_EPOCHS,
        'patience': PATIENCE,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'hidden_dim': HIDDEN_DIM,
        'dropout': DROPOUT,
        'saint_strategy': SAINT_STRATEGY,
        'saint_budget': SAINT_BUDGET,
        'saint_samples_per_epoch': SAINT_SAMPLES_PER_EPOCH,
    },
}

with open(os.path.join(PROCESSED_PATH, 'significance_results.json'), 'w') as f:
    json.dump(output, f, indent=2, default=str)

print("Saved significance_results.json")
print("\nNotebook sequence complete.")

Saved significance_results.json

Notebook sequence complete.
